# Validate `vibe_coder_final_boss`

End-to-end smoke test for the submission. Run on a Colab T4 runtime
(Runtime → Change runtime type → T4 GPU). Total wall time should be
well under the 30-minute eval budget.

**Important:** the eval below runs with `--device cpu`, not `--device cuda`,
even though we have a T4. The reason is the GROUND-TRUTH video decoder:
on CUDA, the eval pipeline uses NVIDIA DALI; on CPU it uses libav. The two
decode paths produce subtly different uint8 frames at the YUV→RGB rounding
stage, and our model is implicitly tuned to libav. Running with `--device cuda`
shifts the GT and inflates seg/pose distortion by ~30-50%, costing ~0.02 score.

We declare 'no GPU required' on the PR so comma's official eval runs on
their CPU runner, matching this notebook.

What this notebook does:
1. Clones the comma_video_compression_challenge repo + LFS pulls the videos.
2. Installs Python deps via `uv` (CPU group).
3. Pulls our submission folder from the BradyMeighan fork.
4. Downloads `archive.zip` from a GitHub Release.
5. Runs `evaluate.sh --device cpu` and prints the report.

If anything fails or the score deviates from `0.23` (rounds from 0.22878), do not submit.

## 1. Clone the challenge repo + pull video LFS

git-lfs is preinstalled on Colab. The videos are ~3 GB.

In [ ]:
%cd /content
!git lfs install
!git clone https://github.com/commaai/comma_video_compression_challenge.git
%cd /content/comma_video_compression_challenge
!git lfs pull
!ls -lh videos/ | head -5

## 2. Install Python deps via `uv`

We use the `cpu` group so torch is CPU-only and no CUDA / DALI are pulled in.
Matches what comma's CPU runner uses. ~1-2 minutes.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
%cd /content/comma_video_compression_challenge
!uv sync --group cpu

## 3. Drop our submission into `submissions/vibe_coder_final_boss/`

Pulled from the BradyMeighan fork.

In [ ]:
%cd /content
!rm -rf fork && git clone https://github.com/BradyMeighan/comma_video_compression_challenge.git fork
!git -C /content/fork checkout submission/vibe_coder_final_boss
!cp -r /content/fork/submissions/vibe_coder_final_boss /content/comma_video_compression_challenge/submissions/
!ls -la /content/comma_video_compression_challenge/submissions/vibe_coder_final_boss/

## 4. Download `archive.zip`

In [ ]:
import os
ARCHIVE_URL = "https://github.com/BradyMeighan/comma_video_compression_challenge/releases/download/vcfb-archive/archive.zip"
DEST = "/content/comma_video_compression_challenge/submissions/vibe_coder_final_boss/archive.zip"
!curl -L -o {DEST} {ARCHIVE_URL}
print("size:", os.path.getsize(DEST), "bytes (expected 197,160)")

## 5. Run `evaluate.sh --device cpu` against our submission

Inflate runs ~4-6 min on CPU. Eval ~1 min. Total ~5-7 min wall time.

In [ ]:
%cd /content/comma_video_compression_challenge
!source .venv/bin/activate && time bash evaluate.sh \
    --submission-dir ./submissions/vibe_coder_final_boss \
    --device cpu 2>&1 | tail -40

## 6. Print the report and double-check the score

Expected: `Final score = 0.23` (rounds from 0.22878). Internal computation:
100 * 0.000272 + sqrt(10 * 0.000495) + 25 * 0.005258 = 0.027 + 0.0703 + 0.1315 = 0.2287.

In [ ]:
!cat /content/comma_video_compression_challenge/submissions/vibe_coder_final_boss/report.txt